<a href="https://colab.research.google.com/github/e22340-SajithaSanthushti/Statistical-Learning-e22340/blob/main/Assignment_7d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

For a random variable following a Beta distribution $\Theta \sim \text{Beta}(\alpha, \beta)$, the analytical mean is given by:$$\mathbb{E}[\Theta] = \frac{\alpha}{\alpha + \beta}$$Given $\alpha = 8$ and $\beta = 1.5$:$$\mathbb{E}[\Theta^{(0)}] = \frac{8}{8 + 1.5} = \frac{8}{9.5} = \frac{16}{19} \approx \mathbf{0.8421}$$2. Engineering Justification for $\text{Beta}(8, 1.5)$Bounded Domain support $[0, 1]$: The Beta distribution naturally constrains values strictly within $(0, 1]$, perfectly matching the physical domain of the remaining stiffness efficiency factor $\theta$ (where $1.0$ is fully intact and values near $0$ represent severe degradation).Left-Skewed / High Mass near 1.0: With parameters $\alpha = 8 > 1$ and $\beta = 1.5 > 1$, the PDF is heavily skewed towards $1.0$. The mode is:$$\text{Mode} = \frac{\alpha - 1}{\alpha + \beta - 2} = \frac{7}{7.5} \approx 0.9333$$This assigns highest probability density to near-pristine conditions, reflecting a reasonable prior belief that a newly deployed component is healthy.Smooth Continuous Falloff: It assigns negligible probability density to values near zero (severe structural failure prior to deployment), yet retains non-zero density across the domain to allow updating via sensor observation data if degradation is present

In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Parameters
alpha_param = 8.0
beta_param = 1.5

# Domain restricted to [0.01, 1.0]
theta = np.linspace(0.01, 1.0, 500)

# Compute Beta PDF
pdf_values = beta.pdf(theta, alpha_param, beta_param)

# Expected Value
mean_val = alpha_param / (alpha_param + beta_param)

# Plot using Plotly
fig = go.Figure()

# Add Prior PDF Line
fig.add_trace(
    go.Scatter(
        x=theta,
        y=pdf_values,
        mode="lines",
        name="Prior PDF: Beta(8, 1.5)",
        line=dict(color="royalblue", width=3),
    )
)

# Add Marker for Expected Value
fig.add_trace(
    go.Scatter(
        x=[mean_val],
        y=[beta.pdf(mean_val, alpha_param, beta_param)],
        mode="markers+text",
        name=f"E[Θ] = {mean_val:.4f}",
        marker=dict(size=10, color="crimson"),
        text=[f"  E[Θ] = {mean_val:.4f}"],
        textposition="top right",
    )
)

# Layout Setup
fig.update_layout(
    title="<b>Initial Prior Distribution: $\\Theta \\sim \\text{Beta}(8, 1.5)$</b>",
    xaxis_title="Stiffness Efficiency Factor (θ)",
    yaxis_title="Probability Density",
    xaxis=dict(range=[0, 1.05], gridcolor="lightgray"),
    yaxis=dict(gridcolor="lightgray"),
    template="plotly_white",
    width=800,
    height=500,
)

fig.show()

From the measurement model, the continuous sensor reading at step $k$ is given by:$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \quad \text{where } \epsilon_k \sim \mathcal{N}(0, \sigma^2)$$Derivation via Properties of the Log-Normal DistributionTaking the natural logarithm of both sides:$$\ln(y_k) = \ln(\theta \cdot K_{\text{nominal}}) + \epsilon_k$$Since $\epsilon_k \sim \mathcal{N}(0, \sigma^2)$, the log-transformed measurement follows a Gaussian distribution:$$\ln(y_k) \sim \mathcal{N}\left(\ln(\theta \cdot K_{\text{nominal}}), \, \sigma^2\right)$$By definition, a variable $Y$ whose logarithm is normally distributed with mean $\mu$ and variance $\sigma^2$ follows a Log-Normal distribution, $\text{LogNormal}(\mu, \sigma^2)$, with probability density function:$$f_Y(y) = \frac{1}{y \sigma \sqrt{2\pi}} \exp\left( -\frac{(\ln(y) - \mu)^2}{2\sigma^2} \right), \quad y > 0$$Substituting $\mu_k = \ln(\theta \cdot K_{\text{nominal}})$, the single measurement likelihood $L(y_k \mid \theta) = f(y_k \mid \theta)$ is:$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right)$$Alternatively, using logarithm rules $\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}}) = \ln\left(\frac{y_k}{\theta \cdot K_{\text{nominal}}}\right)$:$$L(y_k \mid \theta) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{1}{2\sigma^2} \left[ \ln\left(\frac{y_k}{\theta \cdot K_{\text{nominal}}}\right) \right]^2 \right)$$2. Joint Likelihood Function $L(\mathbf{y}^{(k)} \mid \theta)$Assuming the sensor measurement errors $\{\epsilon_1, \epsilon_2, \dots, \epsilon_k\}$ are independent and identically distributed (i.i.d.) conditional on $\theta$, the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of the individual likelihood contributions:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} L(y_i \mid \theta)$$Substituting the single Likelihood expression:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} \left[ \frac{1}{y_i \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\ln(y_i) - \ln(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right) \right]$$Expanding the product into a sum in the exponential term:$$L(\mathbf{y}^{(k)} \mid \theta) = \left(\frac{1}{\sigma \sqrt{2\pi}}\right)^k \left(\prod_{i=1}^{k} \frac{1}{y_i}\right) \exp\left( -\frac{1}{2\sigma^2} \sum_{i=1}^{k} \left[ \ln(y_i) - \ln(\theta \cdot K_{\text{nominal}}) \right]^2 \right)$$

By Bayes' Theorem, the posterior distribution is given by:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{L(\mathbf{y}^{(k)} \mid \theta) \, f_{\Theta}^{(0)}(\theta)}{\int_{0}^{1} L(\mathbf{y}^{(k)} \mid \theta') \, f_{\Theta}^{(0)}(\theta') \, d\theta'}$$An exact closed-form analytical solution requires the prior distribution $f_{\Theta}^{(0)}(\theta)$ and likelihood $L(\mathbf{y}^{(k)} \mid \theta)$ to be conjugate, meaning the posterior distribution belongs to the same parametric family as the prior distribution.Here, a closed-form solution fails to exist due to two main mathematical reasons:Non-Conjugacy of Beta Prior and Log-Normal Likelihood:Prior: $\Theta \sim \text{Beta}(\alpha, \beta)$ has functional dependence $f_{\Theta}^{(0)}(\theta) \propto \theta^{\alpha-1} (1-\theta)^{\beta-1}$.Likelihood: The log-normal likelihood dependence on $\theta$ is embedded within a squared logarithmic exponent:$$L(y_k \mid \theta) \propto \exp\left( -\frac{1}{2\sigma^2} \left[\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})\right]^2 \right)$$Multiplying these two functions yields an expression proportional to:$$\theta^{\alpha-1} (1-\theta)^{\beta-1} \cdot \exp\left( -\frac{1}{2\sigma^2} \left[\ln\left(\frac{y_k}{\theta \cdot K_{\text{nominal}}}\right)\right]^2 \right)$$This functional form does not match any standard probability density function family (such as Beta, Normal, or Gamma).Intractable Evidence Integral (Denominator):The marginal likelihood (normalizing constant) $f(\mathbf{y}^{(k)}) = \int_{0}^{1} L(\mathbf{y}^{(k)} \mid \theta) \, f_{\Theta}^{(0)}(\theta) \, d\theta$ cannot be evaluated analytically because the product of logarithmic-exponential and polynomial factors lacks an antiderivative in terms of elementary or standard special functions.2. Recursive Relationship for Posterior Density at Step $k$Since the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$, Bayes' rule can be formulated recursively:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$Up to a Proportionality Constant ($\propto$):$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right) f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$Omitting factors independent of $\theta$ (such as $1/(y_k \sigma \sqrt{2\pi})$), the simplified proportionality relation is:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \exp\left( -\frac{1}{2\sigma^2} \left[\ln(y_k) - \ln(\theta \cdot K_{\text{nominal}})\right]^2 \right) f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$where the initial condition at step $k=0$ is given by the Beta prior:$$f_{\Theta \mid \mathbf{Y}^{(0)}}(\theta) = f_{\Theta}^{(0)}(\theta) = \frac{\Gamma(\alpha+\beta)}{\Gamma(\alpha)\Gamma(\beta)} \theta^{\alpha-1} (1-\theta)^{\beta-1}, \quad \theta \in [0, 1]$$

To express these point estimators cleanly over the domain $(0, 1]$, let us first define the unnormalized posterior function $g(\theta \mid \mathbf{y}^{(k)})$ as the product of the prior density and the joint likelihood:$$g(\theta \mid \mathbf{y}^{(k)}) = f_{\Theta}^{(0)}(\theta) \cdot L(\mathbf{y}^{(k)} \mid \theta)$$The normalized posterior probability density function is then given by:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \frac{g(\theta \mid \mathbf{y}^{(k)})}{\int_{0}^{1} g(\theta' \mid \mathbf{y}^{(k)}) \, d\theta'}$$1. Running Posterior Mean Estimator ($\hat{\theta}_{\text{Bayes}}^{(k)}$)The Posterior Mean (also known as the Bayes estimate under squared error loss) is calculated as the expected value of $\Theta$ given the running history vector $\mathbf{y}^{(k)}$.Expressed as a ratio of definite integrals over $(0, 1]$:$$\hat{\theta}_{\text{Bayes}}^{(k)} = \mathbb{E}\left[\Theta \mid \mathbf{y}^{(k)}\right] = \int_{0}^{1} \theta \, f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \, d\theta$$Substituting the unnormalized posterior density $g(\theta \mid \mathbf{y}^{(k)})$:$$\hat{\theta}_{\text{Bayes}}^{(k)} = \frac{\int_{0}^{1} \theta \cdot g(\theta \mid \mathbf{y}^{(k)}) \, d\theta}{\int_{0}^{1} g(\theta \mid \mathbf{y}^{(k)}) \, d\theta} = \frac{\int_{0}^{1} \theta \cdot f_{\Theta}^{(0)}(\theta) \prod_{i=1}^{k} L(y_i \mid \theta) \, d\theta}{\int_{0}^{1} f_{\Theta}^{(0)}(\theta) \prod_{i=1}^{k} L(y_i \mid \theta) \, d\theta}$$2. Running Maximum A Posteriori Estimator ($\hat{\theta}_{\text{MAP}}^{(k)}$)The Maximum A Posteriori (MAP) estimate corresponds to the mode of the posterior distribution—i.e., the value of $\theta \in (0, 1]$ that maximizes the posterior density.Since the normalizing integral in the denominator $\int_{0}^{1} g(\theta' \mid \mathbf{y}^{(k)}) \, d\theta'$ is constant with respect to $\theta$, maximizing the normalized posterior density is equivalent to maximizing the unnormalized posterior $g(\theta \mid \mathbf{y}^{(k)})$:$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) = \arg\max_{\theta \in (0, 1]} g(\theta \mid \mathbf{y}^{(k)})$$In terms of the prior and joint likelihood functions:$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg\max_{\theta \in (0, 1]} \left\{ f_{\Theta}^{(0)}(\theta) \cdot \prod_{i=1}^{k} L(y_i \mid \theta) \right\}$$Note on Implementation: In practice, the MAP estimate is typically computed by minimizing the negative log-posterior over the bounded interval $(0, 1]$:$$\hat{\theta}_{\text{MAP}}^{(k)} = \arg\min_{\theta \in (0, 1]} \left\{ -\ln f_{\Theta}^{(0)}(\theta) - \sum_{i=1}^{k} \ln L(y_i \mid \theta) \right\}$$

To maintain the posterior distribution numerically over the bounded domain $\theta \in (0, 1]$, we construct an equally spaced grid of $N$ evaluation points:$$\Theta_{\text{grid}} = [\theta_1, \theta_2, \dots, \theta_N]$$where the step size (grid spacing) is $\Delta \theta = \frac{\theta_N - \theta_1}{N - 1}$.Computational Boundary Handling:Lower Boundary ($\theta \to 0$):Evaluating the log-normal likelihood at $\theta = 0$ results in $\ln(0)$, which causes a numerical division-by-zero or $\text{-inf}$ undefined error.Handling Strategy: Set the lower limit slightly above zero, e.g., $\theta_1 = 0.01$ (or $\theta_1 = 10^{-4}$). This avoids numerical singularities while preserving full physical coverage over realistic damage states.Upper Boundary ($\theta = 1.0$):Set $\theta_N = 1.0$.Evaluating the prior density $\text{Beta}(\alpha, \beta)$ with $\beta = 1.5 > 1$ at $\theta_N = 1.0$ evaluates cleanly to $0$.2. Sequential Numerical Algorithm Step-by-StepStep 0: Initialization ($k = 0$)Discretize the domain: $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_N]^T$.Compute the initial prior density vector $\mathbf{f}^{(0)} = [f_1^{(0)}, f_2^{(0)}, \dots, f_N^{(0)}]^T$ using the continuous Beta prior formula:$$f_j^{(0)} = f_{\Theta}^{(0)}(\theta_j), \quad \text{for } j = 1, 2, \dots, N$$Normalize $\mathbf{f}^{(0)}$ using the trapezoidal rule so that $\int_{0.01}^{1.0} f_{\Theta}^{(0)}(\theta) \, d\theta \approx 1$.Sequential Iteration for Each Inspection Step $k$ ($k = 1, 2, \dots, n$)Upon receiving a new sensor reading $y_k$:Step 1: Compute Likelihood VectorEvaluate the single-measurement log-normal likelihood contribution across all grid points $\boldsymbol{\theta}$:$$L_j = L(y_k \mid \theta_j) = \frac{1}{y_k \sigma \sqrt{2\pi}} \exp\left( -\frac{\left(\ln(y_k) - \ln(\theta_j \cdot K_{\text{nominal}})\right)^2}{2\sigma^2} \right), \quad j = 1, 2, \dots, N$$Log-Sum-Exp Trick for Numerical Stability: To prevent underflow/overflow during long measurement sequences, compute likelihoods in log-space:$$\ln L_j = -\ln(y_k \sigma \sqrt{2\pi}) - \frac{\left(\ln(y_k) - \ln(\theta_j \cdot K_{\text{nominal}})\right)^2}{2\sigma^2}$$Step 2: Unnormalized Posterior UpdateMultiply the previous posterior density vector $\mathbf{f}^{(k-1)}$ element-wise by the new likelihood vector $\mathbf{L}$:$$\tilde{f}_j^{(k)} = L_j \cdot f_j^{(k-1)}, \quad \text{for } j = 1, 2, \dots, N$$Step 3: Sequential Normalization via Composite Trapezoidal RuleTo transform the unnormalized vector $\tilde{\mathbf{f}}^{(k)}$ into a valid PDF probability vector $\mathbf{f}^{(k)}$, calculate the normalizing evidence integral $I_k \approx \int_{\theta_1}^{\theta_N} \tilde{f}^{(k)}(\theta) \, d\theta$ using the composite trapezoidal rule:$$I_k = \frac{\Delta \theta}{2} \left[ \tilde{f}_1^{(k)} + 2 \sum_{j=2}^{N-1} \tilde{f}_j^{(k)} + \tilde{f}_N^{(k)} \right]$$Normalize each grid point to obtain the updated posterior density vector at step $k$:$$f_j^{(k)} = \frac{\tilde{f}_j^{(k)}}{I_k}, \quad \text{for } j = 1, 2, \dots, N$$Step 4: Compute Running Point EstimatesPosterior Mean ($\hat{\theta}_{\text{Bayes}}^{(k)}$): Apply the trapezoidal rule to the weighted vector $\theta_j \cdot f_j^{(k)}$:$$\hat{\theta}_{\text{Bayes}}^{(k)} \approx \frac{\Delta \theta}{2} \left[ \theta_1 f_1^{(k)} + 2 \sum_{j=2}^{N-1} \theta_j f_j^{(k)} + \theta_N f_N^{(k)} \right]$$MAP Estimate ($\hat{\theta}_{\text{MAP}}^{(k)}$): Identify the grid index $j^*$ that maximizes $f_j^{(k)}$:$$j^* = \arg\max_{j \in \{1, \dots, N\}} f_j^{(k)} \implies \hat{\theta}_{\text{MAP}}^{(k)} = \theta_{j^*}$$

In [2]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import beta

# Set seed for reproducible synthetic data generation
np.random.seed(42)

# --- Problem Parameters ---
theta_true = 0.68  # Hidden degradation level after impact
K_nominal = 50.0  # Nominal stiffness (kN/mm)
sigma = 0.15  # Sensor noise standard deviation (log-space)
n_steps = 15  # Number of sequential measurements

# Prior distribution parameters: Beta(8, 1.5)
alpha_prior = 8.0
beta_prior = 1.5

# Grid Discretization over theta in [0.01, 1.0]
N_grid = 1000
theta_grid = np.linspace(0.01, 1.0, N_grid)

# --- Step 1: Simulate Sensor Stream y_k ---
# y_k = theta_true * K_nominal * exp(epsilon_k), epsilon_k ~ N(0, sigma^2)
epsilon = np.random.normal(loc=0.0, scale=sigma, size=n_steps)
y_stream = theta_true * K_nominal * np.exp(epsilon)

# --- Step 2: Initialize Grid and Track Estimates ---
# Initial Prior PDF f^{(0)}(theta)
f_prior = beta.pdf(theta_grid, alpha_prior, beta_prior)
f_prior /= np.trapezoid(f_prior, theta_grid)  # Normalize via trapezoidal rule

# Storage arrays for timeline tracking
posterior_history = {0: f_prior.copy()}
theta_bayes_history = [np.trapezoid(theta_grid * f_prior, theta_grid)]
theta_map_history = [theta_grid[np.argmax(f_prior)]]

# Sequential Bayesian Update Loop
f_current = f_prior.copy()

for k in range(1, n_steps + 1):
    y_k = y_stream[k - 1]

    # Compute single-measurement Log-Normal Likelihood L(y_k | theta)
    # L(y_k | theta) = (1 / (y_k * sigma * sqrt(2*pi))) * exp(-0.5 * ((ln(y_k) - ln(theta * K_nominal)) / sigma)^2)
    log_mean = np.log(theta_grid * K_nominal)
    exponent = -0.5 * ((np.log(y_k) - log_mean) / sigma) ** 2
    likelihood = (1.0 / (y_k * sigma * np.sqrt(2 * np.pi))) * np.exp(exponent)

    # Bayesian Grid Update (Unnormalized)
    f_unnorm = f_current * likelihood

    # Normalization using np.trapezoid
    norm_const = np.trapezoid(f_unnorm, theta_grid)
    f_current = f_unnorm / norm_const

    # Save distribution snapshot
    posterior_history[k] = f_current.copy()

    # Compute Point Estimates
    hat_theta_bayes = np.trapezoid(theta_grid * f_current, theta_grid)
    hat_theta_map = theta_grid[np.argmax(f_current)]

    theta_bayes_history.append(hat_theta_bayes)
    theta_map_history.append(hat_theta_map)

# --- Step 3: Create Plots using Plotly ---
fig = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=(
        "<b>Figure 1: Evolution of Posterior Density Curves</b>",
        "<b>Figure 2: Estimator Convergence Timeline</b>",
    ),
    vertical_spacing=0.15,
)

# Plot 1: Selected Milestones Posterior PDFs k in {0, 1, 2, 5, 10, 15}
milestones = [0, 1, 2, 5, 10, 15]
colors = [
    "#636EFA",
    "#EF553B",
    "#00CC96",
    "#AB63FA",
    "#FFA15A",
    "#19D3F3",
]  # Plotly palette

for k_step, color in zip(milestones, colors):
    label = f"Initial Prior (k=0)" if k_step == 0 else f"Step k={k_step}"
    fig.add_trace(
        go.Scatter(
            x=theta_grid,
            y=posterior_history[k_step],
            mode="lines",
            name=label,
            line=dict(color=color, width=2.5),
        ),
        row=1,
        col=1,
    )

# Vertical line for theta_true in Figure 1
fig.add_vline(
    x=theta_true,
    line_width=2,
    line_dash="dash",
    line_color="black",
    annotation_text="True θ = 0.68",
    annotation_position="top left",
    row=1,
    col=1,
)

# Plot 2: Estimator Convergence vs. True Theta
steps_axis = list(range(n_steps + 1))

fig.add_trace(
    go.Scatter(
        x=steps_axis,
        y=theta_bayes_history,
        mode="lines+markers",
        name="Posterior Mean (θ_Bayes)",
        line=dict(color="crimson", width=3),
        marker=dict(size=7),
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=steps_axis,
        y=theta_map_history,
        mode="lines+markers",
        name="MAP Estimate (θ_MAP)",
        line=dict(color="teal", width=2, dash="dash"),
        marker=dict(size=7),
    ),
    row=2,
    col=1,
)

# Horizontal Reference line for True Theta in Figure 2
fig.add_hline(
    y=theta_true,
    line_width=2,
    line_dash="dot",
    line_color="black",
    annotation_text="True Value (θ = 0.68)",
    annotation_position="bottom right",
    row=2,
    col=1,
)

# Layout adjustments
fig.update_xaxes(
    title_text="Stiffness Efficiency Factor (θ)",
    range=[0.3, 1.02],
    gridcolor="lightgray",
    row=1,
    col=1,
)
fig.update_yaxes(
    title_text="Probability Density", gridcolor="lightgray", row=1, col=1
)

fig.update_xaxes(
    title_text="Inspection Step (k)",
    dtick=1,
    gridcolor="lightgray",
    row=2,
    col=1,
)
fig.update_yaxes(
    title_text="Estimated Efficiency (θ)", gridcolor="lightgray", row=2, col=1
)

fig.update_layout(
    title="<b>Structural Health Monitoring: Sequential Bayesian Grid Updating</b>",
    template="plotly_white",
    height=850,
    width=950,
    legend=dict(x=1.02, y=0.5, bordercolor="Black", borderwidth=1),
)

fig.show()

. Convergence Timeline & Overcoming Initial PriorInitial Bias: At step $k=0$, the system starts with an optimistic "healthy" prior expectation ($\mathbb{E}[\Theta^{(0)}] \approx 0.8421$, $\text{Mode} \approx 0.9333$).Speed of Shift:By $k = 2$ to $k = 3$ sensor readings, the likelihood observations quickly dominate the initial prior mass, causing the density curve to rapidly shift leftward.By $k = 5$, both point estimators ($\hat{\theta}_{\text{Bayes}}$ and $\hat{\theta}_{\text{MAP}}$) closely align near $\theta \approx 0.68$, effectively isolating the damaged state.Between $k = 10$ and $k = 15$, the estimates stably hover within a narrow band around the true value ($\theta_{\text{true}} = 0.68$).

Uncertainty Reduction / Variance Shrinkage:As more continuous sensor measurements arrive, the posterior variance decreases markedly ($Var(\Theta \mid \mathbf{y}^{(k)}) \propto \mathcal{O}(1/k)$). This causes the probability density function to narrow dramatically into a steep spike centered around $0.68$.Actionable Threshold Confidence:In Structural Health Monitoring (SHM), decision thresholds are set for safety alerts (e.g., Alert when $\mathbb{P}(\Theta < 0.70) > 0.95$).A broad density curve (large variance) leaves significant probability mass crossing critical safety thresholds, leading to false alarms or ambiguous risk assessments.As the density curve narrows, the cumulative tail probability becomes sharply defined. Engineers can confidently certify that the component has degraded below operational tolerance, enabling timely maintenance before catastrophic failure occurs.